**Tabla Ventas**

Normalizacion de la columna Ciudad :




In [ ]:
import pandas as pd

# Cargar datos
df = pd.read_csv("ventas_erp_producto_limpio.csv", dtype=str)


def normalizar_ciudad(ciudad):
    if pd.isna(ciudad):
        return None

    # Limpiar formato base
    ciudad = str(ciudad).strip().lower()

    # Diccionario de equivalencias (aquí está la magia)
    mapa_ciudades = {
        "santiago": "Santiago",
        "stgo": "Santiago",
        "santiago de chile": "Santiago",

        "valparaiso": "Valparaiso",
        "valpo": "Valparaiso",

        "concepcion": "Concepcion",

        "la serena": "La Serena",

        "antofagasta": "Antofagasta",

        "temuco": "Temuco",

        "puerto montt": "Puerto Montt",

        "rancagua": "Rancagua"
    }

    # Normalizar
    ciudad = mapa_ciudades.get(ciudad, ciudad.title())

    return ciudad


# Aplicar limpieza
df["ciudad"] = df["ciudad"].apply(normalizar_ciudad)

# Guardar resultado
df.to_csv("ventas_erp_ciudad_limpia.csv", index=False, encoding="utf-8-sig")

print(df["ciudad"].value_counts())

Normalizacion de la columna Fecha:

In [ ]:
import pandas as pd

# Cargar archivo
df = pd.read_csv("ventas_erp_ciudad_limpia.csv", dtype=str)


def normalizar_fecha(fecha):
    if pd.isna(fecha):
        return pd.NaT

    fecha = str(fecha).strip()

    # Intentar varios formatos posibles
    formatos = [
        "%Y-%m-%d",  # 2024-09-25
        "%d/%m/%Y",  # 25/09/2024
        "%m-%d-%Y",  # 09-25-2024
        "%Y/%m/%d",  # 2024/09/25
        "%d-%m-%Y"  # 25-09-2024
    ]

    for formato in formatos:
        try:
            return pd.to_datetime(fecha, format=formato)
        except:
            continue

    # Si no logra convertir, devuelve vacío de fecha
    return pd.NaT


# Aplicar limpieza
df["fecha_venta"] = df["fecha_venta"].apply(normalizar_fecha)

# Opcional: dejarla en formato YYYY-MM-DD
df["fecha_venta"] = df["fecha_venta"].dt.strftime("%Y-%m-%d")

# Guardar archivo limpio
df.to_csv("ventas_erp_fecha_limpia.csv", index=False, encoding="utf-8-sig")

print(df[["fecha_venta"]].head(20))
print("Fechas inválidas:", df["fecha_venta"].isna().sum())

Normalizacion de la columna SKU:

In [ ]:
import pandas as pd
import re

# Leer todo como texto para no perder formato
df = pd.read_csv("ventas_erp_50000.csv", dtype=str)

def normalizar_sku(sku):
    if pd.isna(sku):
        return None

    # Pasar a texto, mayúsculas y quitar espacios extremos
    sku = str(sku).upper().strip()

    # Quitar espacios internos y guiones
    sku = sku.replace(" ", "").replace("-", "")

    # Extraer números
    match = re.search(r'(\d+)', sku)

    if not match:
        return None

    numero = int(match.group(1))

    # Dejar formato estándar
    return f"SKU{numero:05d}"

# Limpiar columna original
df["sku"] = df["sku"].apply(normalizar_sku)

# Guardar archivo limpio
df.to_csv("ventas_erp_50000_limpio.csv", index=False, encoding="utf-8-sig")

print(df[["sku"]].head(20))



Normalizacion de la columna Productos:

In [ ]:
import pandas as pd
import unicodedata

# Cargar datos
df = pd.read_csv("ventas_erp_50000_limpio.csv", dtype=str)


def limpiar_texto(texto):
    if pd.isna(texto):
        return None

    # Pasar a string
    texto = str(texto)

    # Quitar espacios al inicio y final
    texto = texto.strip()

    # Pasar a minúsculas
    texto = texto.lower()

    # Quitar tildes (ej: batería → bateria)
    texto = ''.join(
        c for c in unicodedata.normalize('NFD', texto)
        if unicodedata.category(c) != 'Mn'
    )

    # Reemplazos de negocio (normalización real)
    reemplazos = {
        "notebook": "laptop",
        "celular": "smartphone",
        "powerbank": "power bank",
        "audifonos": "audifonos",
        "parlante": "parlante"
    }

    for k, v in reemplazos.items():
        texto = texto.replace(k, v)

    # Capitalizar cada palabra
    texto = texto.title()

    return texto


# Aplicar limpieza
df["producto"] = df["producto"].apply(limpiar_texto)

# Guardar resultado
df.to_csv("ventas_erp_producto_limpio.csv", index=False, encoding="utf-8-sig")

print(df[["producto"]].head(20))

**Tabla Devoluciones**

Normalizacion de la columna Motivo:

In [ ]:
import pandas as pd
import unicodedata

# Leer Excel
df = pd.read_excel("devoluciones_calidad_2000_limpio5.xlsx", dtype=str)


def quitar_tildes(texto):
    return ''.join(
        c for c in unicodedata.normalize('NFD', texto)
        if unicodedata.category(c) != 'Mn'
    )


def normalizar_motivo(motivo):
    if pd.isna(motivo):
        return None

    # Convertir a texto
    motivo = str(motivo).strip().lower()

    # Quitar tildes
    motivo = quitar_tildes(motivo)

    # Diccionario de equivalencias
    mapa_motivos = {
        "no enciende": "No enciende",
        "pantalla danada": "Pantalla dañada",
        "bateria defectuosa": "Batería defectuosa",
        "no carga": "No carga",
        "falla de audio": "Falla de audio",
        "falla audio": "Falla de audio",
        "boton defectuoso": "Botón defectuoso",
        "sobrecalentamiento": "Sobrecalentamiento",
        "producto incompleto": "Producto incompleto",
        "cable defectuoso": "Cable defectuoso",
        "no sincroniza": "No sincroniza",
        "golpeado": "Golpeado",
        "error de fabrica": "Error de fábrica",
        "error fabrica": "Error de fábrica",
    }

    # Buscar en el mapa; si no existe, dejar bonito
    return mapa_motivos.get(motivo, motivo.title())


# Aplicar limpieza
df["motivo_falla"] = df["motivo_falla"].apply(normalizar_motivo)

# Guardar resultado
df.to_excel("devoluciones_calidad_2000_limpio6.xlsx", index=False)

# Ver resultado
print(df["motivo_falla"].value_counts())

Normalizacion de la columna Fecha:


In [ ]:
import pandas as pd

# Leer Excel
df = pd.read_excel("devoluciones_calidad_2000.xlsx", dtype=str)


def normalizar_fecha(fecha):
    if pd.isna(fecha):
        return pd.NaT

    fecha = str(fecha).strip()

    formatos = [
        "%Y-%m-%d",  # 2024-09-25
        "%d/%m/%Y",  # 25/09/2024
        "%m-%d-%Y",  # 09-25-2024
        "%Y/%m/%d",  # 2024/09/25
        "%d-%m-%Y"  # 25-09-2024
    ]

    for formato in formatos:
        try:
            return pd.to_datetime(fecha, format=formato)
        except:
            continue

    return pd.NaT


# Aplicar limpieza
df["fecha_reclamo"] = df["fecha_reclamo"].apply(normalizar_fecha)

# Dejar todas las fechas en el mismo formato
df["fecha_reclamo"] = df["fecha_reclamo"].dt.strftime("%Y-%m-%d")

# Guardar archivo limpio
df.to_excel("devoluciones_calidad_2000_limpio.xlsx", index=False)

# Mostrar resultado
print(df[["fecha_reclamo"]].head(20))
print("Fechas inválidas:", df["fecha_reclamo"].isna().sum())

Normalizacion de la columna SKU:

In [ ]:
import pandas as pd
import re

# Leer Excel
df = pd.read_excel("devoluciones_calidad_2000_limpio.xlsx", dtype=str)


def normalizar_sku(sku):
    if pd.isna(sku):
        return None

    # Convertir a texto, mayúsculas y limpiar extremos
    sku = str(sku).upper().strip()

    # Quitar espacios y guiones
    sku = sku.replace(" ", "").replace("-", "")

    # Buscar números dentro del texto
    match = re.search(r'(\d+)', sku)

    if not match:
        return None

    numero = int(match.group(1))

    # Formato estándar
    return f"SKU{numero:05d}"


# Aplicar limpieza
df["sku_limpio"] = df["codigo_producto"].apply(normalizar_sku)

# Opcional: reemplazar columna original
df["codigo_producto"] = df["sku_limpio"]

# Guardar archivo limpio
df.to_excel("devoluciones_calidad_2000_limpio5.xlsx", index=False)

# Ver resultado
print(df[["codigo_producto", "sku_limpio"]].head(20))